In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df_complet = pd.read_csv('../data/df_complet.csv', sep=';', dtype={'CODE_INSEE': str})
df_revenu = pd.read_csv('../data/src/revenu.csv', sep=';', dtype={'Code géographique': str})

display(df_complet)
display(df_revenu)

In [ ]:
df_revenu.rename(columns={'Code géographique': 'Code_INSEE'}, inplace=True)
df_complet['Code_INSEE'] = df_complet['Code_INSEE'].astype(str)
df_revenu['Code_INSEE'] = df_revenu['Code_INSEE'].astype(str)

In [ ]:
df_complet.columns

In [ ]:
df = pd.merge(df_complet, df_revenu[['[DISP] Médiane (€)','Code_INSEE']], on=['Code_INSEE'], how='inner')
display(df)

In [ ]:
cols_info = ['Femmes', 'Hommes', 'Agriculteurs',
       'Artisans', 'Cadres', 'Intermédiaires', 'Employés', 'Ouvriers',
       'Retraités', 'Etudiants', 'Inactifs', '15-24 ans',
       '25-39 ans', '40-54 ans', '55-64 ans', '65-79 ans', '80 ans et +',
       'Mariés', 'Pacsés', 'Concubinage', 'Veufs', 'Divorcés', 'Célibataires']

df[cols_info] = df[cols_info].div(df['Population_active'], axis=0) * 100

df = df.rename(columns={col: f'pct_{col}' for col in cols_info})
df.columns

In [ ]:
cols_famille = ['Personne seule', 'Homme seul', 'Femme seule', 'Colocation', 'Famille', 
                'Famille monoparentale', 'Couple sans enfant', 'Couple avec enfants']

df[cols_famille] = df[cols_famille].div(df['Population avec enfants'], axis=0) * 100

df = df.rename(columns={col: f'pct_{col}' for col in cols_famille})
df.columns

In [ ]:
df['mediane_sq'] = df['[DISP] Médiane (€)'] ** 2
df['population_sq'] = df['Population_active'] ** 2

In [ ]:
results = []
features = [col for col in df.columns 
            if col not in ['Année', 'Code_INSEE', 'Résultat']]

for col in features:
    moy_gauche = df[df['Résultat']=='gauche'][col].mean()
    moy_droite = df[df['Résultat']=='droite'][col].mean()
    moy_centre = df[df['Résultat']=='centre'][col].mean()
    
    ecart_max = max(moy_gauche, moy_droite, moy_centre)
    ecart_min = min(moy_gauche, moy_droite, moy_centre)
    ecart = ecart_max - ecart_min
    
    results.append({
        'feature': col,
        'gauche': round(moy_gauche, 3),
        'droite': round(moy_droite, 3),
        'centre': round(moy_centre, 3),
        'ecart': round(ecart, 3)
    })

df_ecarts = pd.DataFrame(results).sort_values('ecart', ascending=False)
print(df_ecarts.to_string())

In [ ]:
df_final = df[[
    'Année',
    'Code_INSEE',
    'Résultat',
    'pct_Retraités', 
    'pct_Cadres', 
    'pct_Mariés', 
    'pct_Ouvriers', 
    'pct_Personne seule', 
    'pct_Famille monoparentale', 
    'pct_Couple sans enfant', 
    'mediane_sq', 
    'pct_Divorcés', 
    'pct_Célibataires',
    'population_sq'
]].copy()

print(df_final['Résultat'].value_counts())

In [ ]:
df_final= df_final[df_final['population_sq']> 100 ** 2]

display(df_final)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

features = [
    'pct_Personne seule',
    'pct_Célibataires',
    'pct_Famille monoparentale',
    'pct_Divorcés',
    'population_sq',
    'pct_Ouvriers',
    'pct_Mariés',
    'pct_Couple sans enfant',
    'pct_Cadres',
    'pct_Retraités',
    'mediane_sq'
]
 
X = df_final[features]
y = df_final['Résultat']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_leaf=5,
    class_weight='balanced_subsample',
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print(classification_report(y_test, y_pred))

# Matrice de confusion
cm = confusion_matrix(y_test, y_pred, labels=['gauche', 'centre', 'droite'])
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=['gauche', 'centre', 'droite'],
            yticklabels=['gauche', 'centre', 'droite'],
            cmap='Blues')
plt.title('Matrice de confusion')
plt.ylabel('Réel')
plt.xlabel('Prédit')
plt.tight_layout()
plt.show()

# Importance des variables
importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
plt.figure(figsize=(14, 6))
importances.plot(kind='bar')
plt.title('Importance des variables')
plt.tight_layout()
plt.show()


In [ ]:
df_final.to_csv('../data/df_model.csv', index=False, sep=';', encoding='utf-8')